In [11]:
#Import openCV
import cv2 as cv

**Function to create the april_tag_identification**

In [12]:
dictionary = cv.aruco.getPredefinedDictionary(cv.aruco.DICT_APRILTAG_36h11)
parameters = cv.aruco.DetectorParameters()
detector = cv.aruco.ArucoDetector(dictionary, parameters)

# IDs of the AprilTags we want to detect
TARGET_IDS = {0, 1, 2, 3, 4, 5, 6, 7}


def april_tag_identification(frame):

    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    #corners, ids, _ = detector.detectMarkers(gray)

    corners, ids, rejected = detector.detectMarkers(gray)

    # print("IDs:", ids)
    # print("Rejected:", len(rejected))

    # Check if any AprilTags were detected
    if ids is not None:

        # Go through each detected AprilTag
        for i, tag_id in enumerate(ids.flatten()):

            # Print the ID of the detected AprilTag
            print("Detected ID:", tag_id)

            # Skip the tag if its ID is not one of our target IDs
            if tag_id not in TARGET_IDS:
                continue

            # Draw the detected AprilTag
            # Important to do i:i+1, as it safely returns an empty list instead of throwing an error
            # aka, doesn't crash the program once it detects an april tag
            # justing having i in ids would raise an IndexError and crash the program otherwise
            cv.aruco.drawDetectedMarkers(
                frame,
                [corners[i]],
                ids[i:i+1]
            )
            print(corners[i])

            


    return frame, corners, ids

**POSE detection:**

In [13]:
import numpy as np

def april_tag_pose(corners, ids, camera_matrix, dist_coeffs, tag_length, frame=None):
    # Dictionary that will hold the pose of every detected tag
    # format: {tag_id: (rvec, tvec)}
    poses = {}

    # If detectMarkers didn't find any tags, ids is None, so there is nothing to estimate
    # returning the empty dictionary early avoids crashing on ids.flatten() below
    if ids is None:
        return poses

    # Pose estimation works by comparing two sets of points:
    # 1. where the tag's corners are in the real world (3D), and
    # 2. where those same corners show up in the image (2D).
    # solvePnP then figures out how the tag must be positioned and rotated
    # relative to the camera for the 3D points to land on the 2D points.

    # Build the 3D corners of the tag in its OWN coordinate system
    # the origin (0, 0, 0) is the center of the tag, and the tag lies flat,
    # so every z value is 0
    half = tag_length / 2

    # The order matters here! detectMarkers returns corners as:
    # top-left, top-right, bottom-right, bottom-left
    # and the 3D points have to follow that exact same order,
    # otherwise the pose will come out flipped or rotated incorrectly.
    obj_points = np.array([
        [-half,  half, 0],
        [ half,  half, 0],
        [ half, -half, 0],
        [-half, -half, 0]
    ], dtype=np.float32)

    # Go through each detected tag
    # enumerate gives back (index, value) pairs, so each entry is a pair
    # entry[0] is the index i, which lines up with corners[i]
    # entry[1] is the actual tag ID at that index
    for entry in enumerate(ids.flatten()):
        i = entry[0]
        tag_id = entry[1]

        # solvePnP (Perspective-n-Point) solves for the tag's pose
        # corners[i] has shape (1, 4, 2), so reshape(4, 2) turns it into
        # a plain list of 4 (x, y) pixel points, which is what solvePnP expects
        # camera_matrix and dist_coeffs come from calibration, and tell openCV
        # how the lens bends the image so the math accounts for it
        # SOLVEPNP_IPPE_SQUARE is built specifically for square markers
        # and needs exactly the 4 corner setup we made above

        # Refer to docs for how this works
        result = cv.solvePnP(
            obj_points,
            corners[i].reshape(4, 2), # .reshpae(4,2) shortens the vector 1, 4, 2 to basically say that there is 1 Tag, 4 Corners, 2 Coordinates per Corner
            camera_matrix,
            dist_coeffs,
            flags=cv.SOLVEPNP_IPPE_SQUARE
        )

        # solvePnP returns three things in order, so we pull each one out by index
        # success = True/False, whether it managed to find a pose
        # rvec = rotation vector, how the tag is rotated relative to the camera
        # tvec = translation vector, where the tag is relative to the camera
        #        (in the same units as tag_length, so meters if you measured in meters)
        success = result[0]
        rvec = result[1]
        tvec = result[2]

        # If solvePnP failed for this tag, skip it instead of storing garbage values
        if not success:
            continue

        # Store the pose using the tag ID as the key
        # int() converts the numpy integer into a regular python integer
        poses[int(tag_id)] = (rvec, tvec)

        # Only draw if a frame was passed in, so the function can also be used
        # purely for math without touching the image
        if frame is not None:
            # drawFrameAxes draws the tag's X (red), Y (green), and Z (blue) axes on the frame
            # the last number is the axis length, here half the size of the tag
            # so the axes are visible but don't take over the image
            cv.drawFrameAxes(frame, camera_matrix, dist_coeffs, rvec, tvec, tag_length * 0.5)

    return poses

Calibration values

In [14]:
# Placeholder calibration values, replace these with real calibration results
# fx and fy are focal lengths in pixels, cx and cy are the image center
# for a 640x480 webcam, a rough guess is fx = fy = 640, cx = 320, cy = 240
camera_matrix = np.array([
    [640.0,   0.0, 320.0],
    [  0.0, 640.0, 240.0],
    [  0.0,   0.0,   1.0]
], dtype=np.float64)

# Zeros means "assume no lens distortion", which is only a rough approximation
dist_coeffs = np.zeros((5, 1))

# Side length of the black square of your printed tag, in meters (5 cm here)
tag_length = 0.05

**Code to run the april_tag_identification function, and POSE Detection:**

In [ ]:
cap = cv.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    #frame = cv.flip(frame, 1)

    # april_tag_identification now returns three things, so we pull each one out by index
    detection_result = april_tag_identification(frame)
    tag_detection = detection_result[0]
    corners = detection_result[1]
    ids = detection_result[2]

    # Estimate the pose of every detected tag and draw the axes on the frame
    # Commented out bit draws the axes
    poses = april_tag_pose(corners, ids, camera_matrix, dist_coeffs, tag_length , frame=tag_detection)

    # Print the pose of each tag
    # poses is a dictionary, so looping over it gives us each tag_id
    for tag_id in poses:
        pose = poses[tag_id]
        rvec = pose[0]
        tvec = pose[1]

        # TODO: Find way to simplify tvec and rvec output so it outputs just x,y,z relative to camera, with a 
        # rotation angle in degrees 
        # Put units on everything 
        
        angle_rad = np.linalg.norm(rvec)
        degrees_rvec = np.degrees(angle_rad)

        # tvec is shape (3, 1), flatten() turns it into a plain [x, y, z]
        # z is the distance straight out from the camera to the tag
        print("Tag", tag_id, "\ndistance (scalar):", np.linalg.norm(tvec), '''"rvec:", rvec.flatten(),''' "\ndegrees of rotation:", degrees_rvec, "tvec (x, y, z):", tvec.flatten(),)

        # Straight-line distance from the camera to the tag
        # this is the 3D version of the pythagorean theorem: sqrt(x^2 + y^2 + z^2)
        distance = np.linalg.norm(tvec)
        print("Tag", tag_id, "distance:", distance)  

    frame = cv.flip(frame, 1)
    cv.imshow('Feed', tag_detection)

    if cv.waitKey(1) & (0xFF == ord('q') or cv.getWindowProperty('Feed', cv.WND_PROP_VISIBLE) < 1):
        break

cap.release()
cv.destroyAllWindows()